# Session 05: HTTP and API Integration

**Notebook 1 of 2: Concepts** · used during the live session and for revision afterwards (about 60 minutes on your own)

Until now, every input came from a file you controlled. From this session on, your code talks to **other people's services**: internal APIs, SaaS platforms and, from Session 09, LLM providers. Network calls fail in ways local code never does. They're slow, they time out, they return errors that mean "try again" and errors that mean "never try again", and they need credentials you must not leak.

By the end of this notebook you will be able to:

1. **Call** REST APIs with HTTPX: `GET` and `POST`, query parameters, JSON bodies and headers, using a reusable `httpx.Client`.
2. **Configure** timeouts, and **handle** status codes and transport errors explicitly, mapping each to a decision.
3. **Load** credentials from environment variables and a `.env` file, without committing or logging them.
4. **Design** a small API client class that returns domain objects, raises domain exceptions and accepts an injectable transport.
5. **Test** HTTP code without a network, with `httpx.MockTransport` and respx.

> **No internet needed.** The next cells start a small fake **Asset Inventory** service on your own machine, in the background, and every example talks to it. One optional cell at the end calls a public service, and only runs if you turn it on. The course environment includes HTTPX and respx; run `uv sync` in the course repository if you haven't since Session 04.

## The engineering problem

The Orbital Logistics (fictional) on-call lead wants the nightly incident report to say, for each breached ticket, **which system** it's about, **who owns it** and **how critical it is**. That information lives in the **Asset Inventory**, an HTTP API run by the infrastructure team.

A first attempt at calling it went wrong in three ways:

1. One night the inventory was being redeployed. The report job waited on a request that never completed, and the job was still "running" at 09:00.
2. A `500` from the inventory crashed the whole report with a long traceback, although the report could have been produced without the enrichment.
3. Someone pasted the inventory token into the script to "get it working", and it's now in the Git history.

All three are about treating the network as unreliable and credentials as secrets. Let's start the fake inventory.

In [ ]:
import json
import threading
import time
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import parse_qs, urlparse

FAKE_TOKEN = "notebook-demo-token"  # a made-up token for this local fake service only
ASSETS = {
    "db-03": {"id": "db-03", "name": "Orders database replica", "type": "database",
              "team": "database", "environment": "production", "criticality": "high"},
    "vpn-ber": {"id": "vpn-ber", "name": "Berlin office VPN concentrator", "type": "network",
                "team": "network", "environment": "production", "criticality": "medium"},
    "dns-int": {"id": "dns-int", "name": "Internal DNS resolvers", "type": "network",
                "team": "network", "environment": "production", "criticality": "high"},
    "badge-zrh": {"id": "badge-zrh", "name": "Zürich depot badge readers", "type": "device",
                  "team": "it-support", "environment": "site", "criticality": "medium"},
}


class FakeInventory(BaseHTTPRequestHandler):
    def reply(self, status, body):
        payload = json.dumps(body, ensure_ascii=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

    def do_GET(self):
        url = urlparse(self.path)
        parts = [part for part in url.path.split("/") if part]
        if self.headers.get("Authorization") != f"Bearer {FAKE_TOKEN}":
            return self.reply(401, {"error": "missing or invalid token"})
        if parts == ["assets"]:
            query = parse_qs(url.query)
            team = query.get("team", [None])[0]
            page, size = int(query.get("page", ["1"])[0]), int(query.get("page_size", ["2"])[0])
            matching = [a for a in ASSETS.values() if team in (None, a["team"])]
            items = matching[(page - 1) * size : page * size]
            return self.reply(200, {"items": items, "page": page, "total": len(matching)})
        if len(parts) == 2 and parts[0] == "assets":
            if parts[1] == "broken-01":
                return self.reply(500, {"error": "internal server error"})
            if parts[1] == "slow-01":
                time.sleep(2)
                return self.reply(200, {"id": "slow-01"})
            if parts[1] in ASSETS:
                return self.reply(200, ASSETS[parts[1]])
            return self.reply(404, {"error": f"asset {parts[1]!r} not found"})
        return self.reply(404, {"error": "no such endpoint"})

    def do_POST(self):
        length = int(self.headers.get("Content-Length") or 0)
        body = json.loads(self.rfile.read(length) or b"{}")
        if self.headers.get("Authorization") != f"Bearer {FAKE_TOKEN}":
            return self.reply(401, {"error": "missing or invalid token"})
        note = {"id": 1, "text": body.get("text"), "ticket_id": body.get("ticket_id")}
        return self.reply(201, note)

    def log_message(self, *args):  # keep the notebook output quiet
        pass


class QuietServer(ThreadingHTTPServer):
    daemon_threads = True

    def handle_error(self, request, client_address):  # clients that time out hang up early
        pass


server = QuietServer(("127.0.0.1", 0), FakeInventory)  # port 0: the OS picks a free port
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE_URL = f"http://127.0.0.1:{server.server_address[1]}"
print("Fake Asset Inventory running at", BASE_URL)

You don't need to study that cell. It's a stand-in for the infrastructure team's real service, and it behaves like one: it needs a token, returns JSON, has a broken asset (`broken-01`) and a slow one (`slow-01`). The workbook ships a fuller version that you run in a terminal.

## Part 1: What happens during one request

You know HTTP: methods, URLs, headers, status codes, JSON bodies. What matters now is **where a request can fail**, because each place fails differently and needs different handling.

![The life of one HTTP request in HTTPX](../../../assets/diagrams/http-request-lifecycle.svg)

**How this works**

- A call such as `client.get(...)` goes through five phases: get a connection (from a pool of open ones, or a new one), connect (DNS, TCP and TLS for `https`), send the request, wait for and read the response, and finally hand you a `Response`.
- Each of the first four phases has its **own timeout**, and failing in each raises a different exception. A `ConnectTimeout` means the server never answered the phone. A `ReadTimeout` means it answered and then went quiet, so it may have *started* processing your request.
- If no response arrives at all, HTTPX raises a `RequestError` subclass. A response with status `500` or `404` is **not** an exception: you get a normal `Response` object, and it's your job to check `status_code` (or call `raise_for_status()`).
- HTTPX applies a **5-second timeout** to every phase by default. The popular `requests` library has **no** default timeout, which is how a job ends up waiting all night, like the first failure in the engineering problem.
- The exception tree at the bottom tells you what to catch: `httpx.TimeoutException` for any timeout, `httpx.TransportError` for any network failure, and `httpx.HTTPStatusError` only if you use `raise_for_status()`.

## Part 2: HTTPX basics

HTTPX has two styles. Module-level functions such as `httpx.get()` are fine for a one-off call. **Predict:** what status code and body will this return?

In [ ]:
import httpx

response = httpx.get(f"{BASE_URL}/assets/db-03")
print(response.status_code)
print(response.json())

**Explain:** `401`. The service wants a token, and we didn't send one. No exception was raised: the request worked perfectly at the HTTP level, it just got a "no" back. Status codes are data until you decide otherwise.

For real code, create one **`httpx.Client`** and reuse it. It holds the settings every request shares (base URL, headers, timeout) and keeps connections open between requests, which is much faster than connecting every time.

In [ ]:
with httpx.Client(
    base_url=BASE_URL,
    headers={"Authorization": f"Bearer {FAKE_TOKEN}", "Accept": "application/json"},
    timeout=5.0,
) as client:
    response = client.get("/assets/db-03")
    print(response.status_code, response.headers["content-type"])
    asset = response.json()
    print(asset["name"], "| owned by", asset["team"], "| criticality", asset["criticality"])

    page = client.get("/assets", params={"team": "network", "page": 1, "page_size": 5})
    print(page.request.url)          # params are encoded for you
    print([a["id"] for a in page.json()["items"]])

    note = {"text": "Disk replaced", "ticket_id": "INC-3004"}
    created = client.post("/assets/db-03/notes", json=note)
    print(created.status_code, created.json())

Things to notice:

- `base_url` plus a relative path (`"/assets/db-03"`) keeps the address in one place.
- `params=` builds the query string, including escaping, so never glue it together with f-strings.
- `json=` serialises the body and sets `Content-Type: application/json`. For the response, `.json()` parses the body; `.text` gives the raw string.
- `with httpx.Client(...) as client:` closes the connections afterwards (Session 03's context managers).

| Task | Python (HTTPX) | Java (`java.net.http`) | C# (`HttpClient`) | TypeScript (`fetch`) |
|---|---|---|---|---|
| Shared client | `httpx.Client(base_url=..., headers=...)` | `HttpClient.newBuilder()...build()` | one `HttpClient`, via `IHttpClientFactory` | (none; module-level `fetch`) |
| GET with query | `client.get("/x", params={...})` | `HttpRequest` with a built URI | `GetAsync($"/x?{query}")` | `fetch(url + "?" + new URLSearchParams(...))` |
| POST JSON | `client.post("/x", json=body)` | `BodyPublishers.ofString(json)` | `PostAsJsonAsync("/x", body)` | `fetch(url, {method: "POST", body: JSON.stringify(...)})` |
| Default timeout | 5 seconds | none | 100 seconds | none |
| 4xx/5xx raise? | no; `raise_for_status()` | no | no; `EnsureSuccessStatusCode()` | no; check `response.ok` |

**Why HTTPX and not `requests`?** You'll see `requests` everywhere, and its API is nearly identical. HTTPX adds default timeouts and an **async** client with the same interface, which Session 08 uses, plus first-class support for test transports, which you'll use in Part 6.

## Part 3: When things go wrong

### Status codes are decisions

Not every error deserves the same reaction. Decide per status code, before writing any `except`:

| Status | Meaning | What the client should do |
|---|---|---|
| `200`, `201`, `204` | Success | Parse and return the result |
| `400`, `422` | Your request was invalid | Don't retry; it will fail again. A bug, or bad input: raise a clear error |
| `401`, `403` | Token missing, wrong or not allowed | Don't retry; it's a configuration problem. Raise an "auth" error that never contains the token |
| `404` | It doesn't exist | Often not an error at all, but a normal answer: raise "not found" or return `None` |
| `429` | Too many requests | Retry later; the `Retry-After` header often says when |
| `500`, `502`, `503`, `504` | The service failed | Possibly temporary. Report "unavailable"; retrying can make sense for reads |

`raise_for_status()` turns any 4xx or 5xx into `httpx.HTTPStatusError`. It's convenient in scripts, but in a client you usually want the finer distinctions above. **Predict:** what does this print?

In [ ]:
with httpx.Client(base_url=BASE_URL, headers={"Authorization": f"Bearer {FAKE_TOKEN}"}) as client:
    for asset_id in ["db-03", "nope", "broken-01"]:
        response = client.get(f"/assets/{asset_id}")
        try:
            response.raise_for_status()
            print(asset_id, "->", response.json()["name"])
        except httpx.HTTPStatusError as error:
            print(asset_id, "->", error.response.status_code, error.response.json()["error"])

### Transport failures: timeouts and refused connections

**Predict:** `slow-01` takes 2 seconds to answer. What happens with a 0.5-second timeout? And with a server that isn't running at all (port 9 is almost certainly closed)?

In [ ]:
AUTH = {"Authorization": f"Bearer {FAKE_TOKEN}"}

with httpx.Client(base_url=BASE_URL, headers=AUTH, timeout=0.5) as client:
    started = time.perf_counter()
    try:
        client.get("/assets/slow-01")
    except httpx.TimeoutException as error:
        print(f"{type(error).__name__} after {time.perf_counter() - started:.2f}s: {error}")

try:
    httpx.get("http://127.0.0.1:9/assets/db-03", timeout=5)
except httpx.TransportError as error:
    print(f"{type(error).__name__}: {error}")

**Explain:** the slow call gave up after about half a second with a `ReadTimeout`; the connection worked, but the answer didn't come in time. The second call failed with `ConnectError`, because nothing was listening. (On Linux and macOS that's instant. Windows retries a refused connection for a second or two, so with a very short timeout you'd see `ConnectTimeout` instead.) Both are `TransportError`s: there was no response to inspect.

**Engineering Insight:** choose timeouts deliberately. Five seconds is a sane default for an interactive tool. For a nightly batch, a slightly longer read timeout may be fine, but **never "no timeout"**. A missing timeout turns one stuck server into a stuck job, and a stuck job into a 09:00 incident.

**Common Mistake:** `except Exception:` around HTTP calls. It hides bugs in your own code (a `KeyError` from a wrong field name) behind "the service is down". Catch `httpx.TransportError` for network problems, and check status codes explicitly.

## Part 4: Credentials and configuration

The token belongs to the **environment**, not the code. The standard pattern:

1. Code reads `os.environ["INVENTORY_API_TOKEN"]`.
2. On a developer machine, a `.env` file (git-ignored) holds the values, loaded with **python-dotenv**.
3. A committed `.env.example` documents every variable with placeholder values.
4. In CI and production, the platform injects real environment variables (secrets managers, CI secrets), and there's no `.env` file at all.

You met this pattern in Session 01, Part 8, for LLM keys. Let's look closely at how `load_dotenv` behaves. **Predict:** after this cell, which value does `INVENTORY_API_TOKEN` have?

In [ ]:
import os
import tempfile
from pathlib import Path

from dotenv import load_dotenv

with tempfile.TemporaryDirectory() as workdir:
    env_file = Path(workdir) / ".env"
    env_file.write_text(
        "INVENTORY_BASE_URL=http://127.0.0.1:8765\nINVENTORY_API_TOKEN=from-dotenv-file\n",
        encoding="utf-8",
    )
    os.environ["INVENTORY_API_TOKEN"] = "already-in-environment"

    load_dotenv(env_file)

    print("INVENTORY_BASE_URL  =", os.environ["INVENTORY_BASE_URL"])
    print("INVENTORY_API_TOKEN =", os.environ["INVENTORY_API_TOKEN"])

# Tidy up, so this demo leaves no hidden state in the notebook.
for name in ["INVENTORY_BASE_URL", "INVENTORY_API_TOKEN"]:
    os.environ.pop(name, None)

**Explain:** the base URL came from the file, but the token kept the value that was **already** in the environment. `load_dotenv` never overrides existing variables by default. That's deliberate: in CI or production, the real environment must win over a stray `.env`. (`override=True` reverses it; you rarely want that.)

(Printing a token, as this cell does, is acceptable here only because it's a made-up value in a throwaway demo. Never do it with a real one.)

### Keeping secrets out of output

Secrets leak through **logs, error messages and tracebacks** far more often than through code. One cheap defence: keep the token out of your settings object's `repr`, so it can't appear in a log line or a debugger view by accident.

In [ ]:
from dataclasses import dataclass, field


@dataclass(frozen=True)
class Settings:
    base_url: str
    token: str = field(repr=False)
    timeout_seconds: float = 5.0


settings = Settings(base_url=BASE_URL, token=FAKE_TOKEN)
print(settings)                          # the token is not shown
print(f"configured: {settings!r}")       # nor in f-strings that use repr

| Do | Don't |
|---|---|
| Read secrets from the environment; document them in `.env.example` | Put real values in code, notebooks, `.env.example` or test fixtures |
| Add `.env` to `.gitignore` before creating it | `git add .` in a folder with an un-ignored `.env` |
| Mention *which* variable is missing in errors: "INVENTORY_API_TOKEN is not set" | Echo its value: "token abc123 was rejected" |
| Revoke a token that was ever committed, even after deleting it | Assume deleting the file removed it from Git history |

Session 01, Part 8 applied the same rules to LLM provider keys, which you'll use from Session 09.

## Part 5: Designing an API client

Scattering `httpx.get(...)` calls through application code spreads HTTP details everywhere: URLs, headers, status-code checks. Instead, wrap the API in **one small class** that speaks the language of the business.

![Layers of an API client](../../../assets/diagrams/api-client-architecture.svg)

**How this works**

- Application code calls `client.get_asset("db-03")` and gets an `Asset` back, or a domain exception such as `AssetNotFound`. It never sees status codes, JSON or HTTPX exceptions, so the report code stays readable and testable.
- `InventoryClient` is the single place that knows the API: paths, query parameters, pagination, and what each status code means. When the API changes, only this class changes.
- Underneath, the client owns **one** `httpx.Client`, which carries the base URL, the auth header, the timeout and the connection pool. It's closed when the client is used in a `with` block.
- The **transport** at the bottom is injectable. In production it's the real network; in tests it's `MockTransport` or respx, so the same client code runs against canned responses.
- Configuration flows in from the environment through `load_settings()`. The token travels into the `Authorization` header, and nowhere else.

Here is a minimal version with one method. The workbook asks you to build the full one.

In [ ]:
class InventoryError(Exception):
    """The inventory didn't give a usable answer."""


class AssetNotFound(InventoryError):
    """The asset doesn't exist."""


class InventoryUnavailable(InventoryError):
    """Timeout, connection failure or 5xx."""


class MiniInventoryClient:
    def __init__(self, base_url, token, *, timeout_seconds=5.0, transport=None):
        self._http = httpx.Client(
            base_url=base_url,
            headers={"Authorization": f"Bearer {token}"},
            timeout=timeout_seconds,
            transport=transport,
        )

    def __enter__(self):
        return self

    def __exit__(self, *exc_info):
        self._http.close()

    def get_asset(self, asset_id: str) -> dict:
        try:
            response = self._http.get(f"/assets/{asset_id}")
        except httpx.TransportError as error:
            raise InventoryUnavailable(f"GET /assets/{asset_id} failed: {error}") from error
        if response.status_code == 404:
            raise AssetNotFound(asset_id)
        if response.status_code >= 500:
            raise InventoryUnavailable(f"inventory returned {response.status_code}")
        if response.status_code != 200:
            raise InventoryError(f"unexpected status {response.status_code}")
        return response.json()


with MiniInventoryClient(BASE_URL, FAKE_TOKEN, timeout_seconds=0.5) as inventory:
    print(inventory.get_asset("badge-zrh")["name"])
    for asset_id in ["nope", "broken-01", "slow-01"]:
        try:
            inventory.get_asset(asset_id)
        except InventoryError as error:
            print(f"{asset_id}: {type(error).__name__}: {error}")

The calling code now handles **three meanings**, not dozens of HTTP details. For the nightly report, `InventoryUnavailable` means "produce the report without enrichment and say so", which fixes the second failure in the engineering problem. Note that the timeout's `ReadTimeout` is a `TransportError`, so it becomes `InventoryUnavailable` with the original attached as `__cause__`, exactly the chaining pattern from Session 03.

### Debugging: why is the enrichment so slow?

A colleague's version creates a new client for every lookup. **Predict:** how much slower is that than reusing one client, for 20 lookups?

In [ ]:
def lookup_with_new_client_each_time(n):
    for _ in range(n):
        with httpx.Client(base_url=BASE_URL, headers=AUTH) as c:
            c.get("/assets/db-03")


def lookup_with_one_client(n):
    with httpx.Client(base_url=BASE_URL, headers=AUTH) as c:
        for _ in range(n):
            c.get("/assets/db-03")


for strategy in [lookup_with_new_client_each_time, lookup_with_one_client]:
    started = time.perf_counter()
    strategy(20)
    print(f"{strategy.__name__:<36} {time.perf_counter() - started:.3f}s")

**Explain:** your numbers depend on your machine, but reusing one client should be clearly faster, even against a server on the same computer. A new client means a new connection pool, so every request pays for a fresh TCP connection. Against a real `https` service on another continent, each new connection also costs DNS and a TLS handshake: often 100 ms or more, per request. Create the client once, and pass it around (or wrap it, as `InventoryClient` does).

## Part 6: Testing without a network

Tests that call real services are slow, flaky, and can't easily produce a `503` or a timeout on demand. Replace the **transport** instead.

### `httpx.MockTransport`

A `MockTransport` wraps a function that receives each `httpx.Request` and returns an `httpx.Response`. Because our client accepts `transport=`, we can test it with no server at all:

In [ ]:
requests_seen = []


def fake_inventory(request: httpx.Request) -> httpx.Response:
    requests_seen.append((request.method, request.url.path, request.headers["Authorization"]))
    if request.url.path == "/assets/db-03":
        return httpx.Response(200, json={"id": "db-03", "name": "Orders database replica"})
    if request.url.path == "/assets/down-01":
        raise httpx.ConnectTimeout("simulated: no answer", request=request)
    return httpx.Response(404, json={"error": "not found"})


transport = httpx.MockTransport(fake_inventory)
with MiniInventoryClient("http://inventory.test", "test-token", transport=transport) as inventory:
    assert inventory.get_asset("db-03")["name"] == "Orders database replica"
    for asset_id, expected in [("nope", AssetNotFound), ("down-01", InventoryUnavailable)]:
        try:
            inventory.get_asset(asset_id)
        except expected as error:
            print(f"{asset_id}: raised {type(error).__name__} as expected")

print(requests_seen)

`http://inventory.test` doesn't exist, and doesn't need to: the request never leaves the process. Recording the requests lets a test check **what was sent** (path, headers, body), not just what came back.

### respx

**respx** builds on the same idea with a routing API, so you describe expected requests declaratively. It patches HTTPX itself, so it works even when you can't inject a transport. Inside `respx.mock`, any request that matches no route fails the test, a useful guard against calling something you didn't intend to.

In [ ]:
import respx

with respx.mock(base_url="http://inventory.test") as router:
    asset_json = {"id": "db-03", "name": "Orders database replica"}
    asset_route = router.get("/assets/db-03").respond(200, json=asset_json)
    router.get("/assets/broken-01").respond(503)
    router.get("/assets/slow-01").mock(side_effect=httpx.ReadTimeout("simulated slow server"))

    with MiniInventoryClient("http://inventory.test", "test-token") as inventory:
        print(inventory.get_asset("db-03")["name"])
        for asset_id in ["broken-01", "slow-01"]:
            try:
                inventory.get_asset(asset_id)
            except InventoryUnavailable as error:
                cause = type(error.__cause__).__name__ if error.__cause__ else None
                print(f"{asset_id}: {error}  (cause: {cause})")

    print("route called:", asset_route.called, "| times:", asset_route.call_count)
    print("Authorization sent:", asset_route.calls.last.request.headers["Authorization"])

In pytest, respx provides a ready-made fixture called `respx_mock`, and the workbook's `conftest.py` defines an `inventory` fixture with a base URL. A typical test:

```python
def test_unknown_asset(inventory, client):
    inventory.get("/assets/nope").respond(404, json={"error": "not found"})

    with pytest.raises(AssetNotFound):
        client.get_asset("nope")
```

| Need | MockTransport | respx |
|---|---|---|
| No extra dependency | ✓ (part of HTTPX) | separate package |
| Requires injecting the transport | yes | no (patches HTTPX) |
| Declarative routes, call assertions | write them yourself | built in |
| Best for | small fakes, and as a design check that your client is injectable | most client test suites |

**What to test in an API client:** the happy path, and *each* row of the status-code table in Part 3, plus timeouts and connection errors. Check that the right request was sent (path, parameters, headers, body), and that secrets never appear in error messages.

## Part 7: Retries (awareness)

Some failures are temporary: a `503` during a deployment, a connection reset, a `429`. Retrying them can turn a failure into a success. Retrying the wrong thing makes things worse.

- **Safe to retry:** idempotent requests (`GET`, and `PUT`/`DELETE` by design) after timeouts, connection errors, `429`, `502`, `503` and `504`.
- **Not safe to retry blindly:** `POST`. If a `ReadTimeout` hit *after* the server created a note, retrying creates a second one. Use an idempotency key, if the API supports one.
- **Never retry:** `400`, `401`, `403`, `404`, `422`. The answer won't change.
- **Back off** between attempts (for example 0.5 s, 1 s, 2 s) and give up after a few, so that a struggling service isn't hammered by every client at once.

`httpx.HTTPTransport(retries=2)` retries only failed **connection attempts**, nothing else. For anything more, a small helper is often enough. Libraries such as `tenacity` and `stamina` package the pattern with jitter and logging.

In [ ]:
def get_with_retries(client: httpx.Client, path: str, *, attempts: int = 3, backoff: float = 0.1):
    """GET with retries for temporary failures only. Returns the final response."""
    for attempt in range(1, attempts + 1):
        try:
            response = client.get(path)
            if response.status_code not in (429, 502, 503, 504):
                return response
        except httpx.TransportError:
            if attempt == attempts:
                raise
        if attempt < attempts:
            time.sleep(backoff * 2 ** (attempt - 1))   # 0.1 s, then 0.2 s, ...
    return response


calls = {"n": 0}


def flaky(request):
    calls["n"] += 1
    return httpx.Response(503) if calls["n"] < 3 else httpx.Response(200, json={"ok": True})


with httpx.Client(base_url="http://inventory.test", transport=httpx.MockTransport(flaky)) as client:
    final = get_with_retries(client, "/assets/db-03")
    print(final.status_code, "after", calls["n"], "attempts")

For this course, the rule is simple: **always set a timeout, map errors deliberately, and retry only idempotent reads.** Session 08 revisits resilience when many requests run concurrently.

## Optional: one real request over the internet

Everything above ran against your own machine. If you'd like to see a public API respond, set `RUN_LIVE_CALL = True` and run the cell. It calls [httpbin.org](https://httpbin.org), a free service that echoes requests back, and sends no credentials. Leave it `False` if you're offline or behind a restrictive proxy.

In [ ]:
RUN_LIVE_CALL = False

if RUN_LIVE_CALL:
    try:
        params = {"course": "python-ai-bridge"}
        live = httpx.get("https://httpbin.org/get", params=params, timeout=10)
        live.raise_for_status()
        print(live.json()["args"], "from", live.json()["url"])
    except httpx.HTTPError as error:
        print(f"Live call failed ({type(error).__name__}): {error}.")
        print("Everything else in this notebook still works offline.")
else:
    print("Skipped. Set RUN_LIVE_CALL = True to try a real request.")

## Common mistakes

| Mistake | What happens | Instead |
|---|---|---|
| No timeout (common with `requests`) | One stuck server hangs your job indefinitely | Always set one; HTTPX defaults to 5 s |
| A new client for every request | A new connection each time: slow, and wasteful for the server | One `httpx.Client`, reused and closed with `with` |
| Assuming a response means success | A `500` page is parsed as data, or `.json()` fails on HTML | Check `status_code`, or call `raise_for_status()` |
| `except Exception:` around HTTP calls | Your own bugs look like "service down" | Catch `httpx.TransportError`; branch on status codes |
| Building query strings with f-strings | Broken URLs when values contain `&`, spaces or `#` | `params={...}` |
| Retrying everything | Duplicate POSTs; hammering a failing service | Retry only idempotent requests and temporary failures, with backoff |
| Token in code, `.env.example`, logs or error messages | Leaked credentials | Environment variables; `field(repr=False)`; name the variable, never the value |
| Tests that call the real service | Slow, flaky, and can't simulate failures | `MockTransport` or respx |

## Practical engineering application

Every integration from here on has this shape. Session 06 validates API responses with Pydantic instead of hand-written checks. Session 07's FastAPI service calls the inventory through this client. Session 08 makes many calls concurrently with `httpx.AsyncClient`. And Session 09's LLM SDKs are API clients too: HTTPX underneath, with timeouts, retries, status codes and secret keys, all the concepts you now know.

## Key takeaways

- A response with a `4xx` or `5xx` status is not an exception; no response at all is (`httpx.RequestError`).
- Each phase has a timeout. Never run without one.
- Map status codes to decisions: 404 is often a normal answer, 401 is configuration, 5xx and timeouts are "unavailable".
- Secrets live in the environment, are documented in `.env.example`, and never appear in code, logs or `repr`.
- Wrap an API in one client class: domain methods in, domain objects and domain exceptions out, one reused `httpx.Client`, an injectable transport.
- Test HTTP code with `MockTransport` or respx, covering every failure mode, without a network.

## Further reading

The curated references for this session are in [`../resources/references.md`](../resources/references.md).

**Next:** test yourself with **`02-practice-quiz.ipynb`**, then start the workbook.